# Phase 3: Export and CPU Latency Benchmark
YOLO26n vs YOLO11n, ONNX export, CPU-only inference speed

**Input needed:** none new. This reads the `best.pt` weights Phase 2 already trained. Run this in the **same session** right after Phase 2, if you start a fresh notebook the weights won't be on disk and Phase 2 has to run again first.

**Why CPU-only:** the deployed demo (Phase 4) runs on Hugging Face's free CPU tier, not a GPU. This benchmark forces `CPUExecutionProvider` explicitly, so it measures the number that actually matters for the demo, regardless of whether a GPU is present in this session.

In [1]:
# safe to rerun even if already installed
!pip install -q ultralytics onnxruntime

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 5.1 MB/s eta 0:00:00


In [2]:
"""
Locate Phase 2's best.pt for each model without hardcoding the exact save path.
Searches both the working directory and /kaggle/input, since weights may be in
a live session or attached as a dataset.
"""
from pathlib import Path


def find_best_weights(model_dir_name: str, search_roots: tuple[str, ...] = (".", "/kaggle/input")) -> Path:
    matches = []
    for root in search_roots:
        if Path(root).exists():
            matches.extend(Path(root).rglob(f"{model_dir_name}/weights/best.pt"))
    matches = sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(
            f"No best.pt found under a '{model_dir_name}/weights' folder in {search_roots}. "
            "Check the dataset is attached via Add Data."
        )
    return matches[0]


for name in ["yolo26n", "yolo11n"]:
    print(f"{name}: {find_best_weights(name)}")

yolo26n: /kaggle/input/datasets/daudshah/phase2-full-training-output/runs/detect/runs/phase2_full/yolo26n/weights/best.pt
yolo11n: /kaggle/input/datasets/daudshah/phase2-full-training-output/runs/detect/runs/phase2_full/yolo11n/weights/best.pt


In [3]:
"""Export both Phase 2 models to ONNX."""
import shutil
from ultralytics import YOLO

WRITABLE_DIR = Path("/kaggle/working/weights")
WRITABLE_DIR.mkdir(parents=True, exist_ok=True)

onnx_paths: dict[str, str] = {}

for name in ["yolo26n", "yolo11n"]:
    weights = find_best_weights(name)
    # export writes best.onnx next to the source .pt file, /kaggle/input is
    # read-only, so copy to a writable location first
    local_weights = WRITABLE_DIR / f"{name}_best.pt"
    shutil.copy(weights, local_weights)
    try:
        model = YOLO(str(local_weights))
        exported = model.export(format="onnx", imgsz=640, simplify=True)
    except Exception as e:
        raise RuntimeError(f"{name} ONNX export failed: {e}") from e
    onnx_paths[name] = str(exported)
    print(f"{name}: exported to {exported}")

print(onnx_paths)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
Ultralytics 8.4.127 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 122 layers, 2,376,786 parameters, 0 gradients, 5.3 GFLOPs

PyTorch: starting from '/kaggle/working/weights/yolo26n_best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)
requirements: Ultralytics requirement ['onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 10 packages in 334ms
Prepared 1 package in 78ms
Installed 1 package in 11ms
 + onnxslim

In [4]:
"""Benchmark CPU-only inference latency for both exported ONNX models."""
import time
import numpy as np
import onnxruntime as ort


def benchmark_onnx(model_path: str, imgsz: int = 640, runs: int = 100) -> dict[str, float]:
    """Runs `runs` forward passes on random input, returns latency stats in ms."""
    session = ort.InferenceSession(model_path, providers=["CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name
    dummy = np.random.rand(1, 3, imgsz, imgsz).astype(np.float32)

    # warmup, the first call pays a one-time init cost that would skew the average
    for _ in range(5):
        session.run(None, {input_name: dummy})

    times = []
    for _ in range(runs):
        start = time.perf_counter()
        session.run(None, {input_name: dummy})
        times.append((time.perf_counter() - start) * 1000)

    times = np.array(times)
    return {
        "mean_ms": float(times.mean()),
        "p95_ms": float(np.percentile(times, 95)),
        "fps": float(1000 / times.mean()),
    }


latency_results: dict[str, dict[str, float]] = {}
for name, path in onnx_paths.items():
    stats = benchmark_onnx(path)
    latency_results[name] = stats
    print(f"{name}: {stats['mean_ms']:.1f} ms mean, {stats['p95_ms']:.1f} ms p95, {stats['fps']:.1f} FPS")

yolo26n: 69.9 ms mean, 77.3 ms p95, 14.3 FPS
yolo11n: 82.4 ms mean, 98.0 ms p95, 12.1 FPS


In [5]:
"""Combine Phase 2 accuracy and Phase 3 latency into one final comparison table, ready to paste into the README or LinkedIn post."""
import pandas as pd


def find_results_csv(model_dir_name: str, search_roots: tuple[str, ...] = (".", "/kaggle/input")) -> Path:
    matches = []
    for root in search_roots:
        if Path(root).exists():
            matches.extend(Path(root).rglob(f"{model_dir_name}/results.csv"))
    matches = sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)
    if not matches:
        raise FileNotFoundError(f"No results.csv found for '{model_dir_name}' in {search_roots}, run Phase 2 first.")
    return matches[0]


def accuracy_metrics(model_dir_name: str) -> dict[str, float]:
    df = pd.read_csv(find_results_csv(model_dir_name))
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]
    return {
        "mAP50": float(last.get("metrics/mAP50(B)", float("nan"))),
        "mAP50-95": float(last.get("metrics/mAP50-95(B)", float("nan"))),
    }


print(f"{'Model':<10}{'mAP50':>8}{'mAP50-95':>10}{'CPU ms':>10}{'FPS':>8}")
for name in ["yolo26n", "yolo11n"]:
    acc = accuracy_metrics(name)
    lat = latency_results[name]
    print(f"{name:<10}{acc['mAP50']:>8.3f}{acc['mAP50-95']:>10.3f}{lat['mean_ms']:>10.1f}{lat['fps']:>8.1f}")

Model        mAP50  mAP50-95    CPU ms     FPS
yolo26n      0.263     0.145      69.9    14.3
yolo11n      0.280     0.158      82.4    12.1


## Phase 3 is complete when:
- Both models exported without error
- The latency cell prints real (non-zero) ms and FPS for both
- The final table shows both accuracy and latency side by side

This is the real test of YOLO26's headline claim, CPU inference speed, not training accuracy. Report whatever the numbers actually show, including if YOLO26 doesn't win here either.

**Next: Phase 4**, Gradio demo deployed to HF Spaces, using whichever ONNX model you decide to ship.